# Lab 2: benchmark three vector databases and defend one

**Time: about 80 minutes. Work in pairs.** Same rhythm as Lab 1: look, write your number, then reveal.

**What you will build.** You put the same data into three real vector databases, **Chroma**, **Qdrant**
and **Milvus**, measure how much each one finds and how long it takes, turn the one dial they share to
draw a curve, see what compression costs, test what a filter does to your results, measure the
trainer's Databricks AI Search index the same way, and finish with one line you could defend to an
architect: which database, at which setting, against which target.

**Why these three.** They are among the best-known vector databases (GitHub stars and Stack Overflow's
2025 survey, 30 September 2026), and all three run inside this notebook: no server, no Docker, no
account. Each builds a real index with a search effort setting, so the trade between recall and speed
actually shows up.

**How the cells work.** Same as Lab 1. The cells marked `YOUR CALL` are the only ones you edit, and
each already holds a working value.

**Four words used throughout.** A *vector* is the list of numbers that stands for one piece of text.
*Recall at 10* is the share of the ten correct results that a database actually returned, so 1.0 means
it missed nothing. *p50* is the middle response time and *p95* is the slow end, the time that 95
questions out of 100 beat. *ef* is the search effort: how many candidates the search keeps in hand
while it hops through the index. More ef, more recall, more time.

In [ ]:
# Setup. This cell is the same in every lab notebook and has to run first.
import sys, pathlib, textwrap

# ROOT must point at the lab folder. You may have opened this notebook from the lab
# folder or from notebooks/, so check which, and step up one level if needed.
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# src/ holds the shared lab code. Putting it on the search path is what makes
# `from freshcart import ...` work in the cells below.
sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "data" / "raw"     # the seven original documents. Nothing ever writes here.
print("lab folder:", ROOT)
print("corpus files:", len(list(RAW.glob("*"))))

In [ ]:
# The three databases and the helpers that measure them live in src/vector_dbs.py:
#   open_db(name)        Chroma, Qdrant or Milvus, running inside this notebook
#   make_benchmark_set   100,000 vectors and 200 questions, plus the exact answer key
#   measure, sweep       recall and response time, at one setting or across the dial
#   filter_test          what a filter does to the number of results you get back
#   compression_test     the same index stored smaller, and what that costs
import numpy as np
from freshcart import load_all, chunk_docs, get_embedder
from stores import FlatStore, ground_truth
from labkit import table
import vector_dbs as vdb

QUERIES = [
    "how many refund claims are allowed in thirty days",
    "who pays when packaging causes a spill",
    "spillage photographic evidence deadline",
    "when must a case be escalated to a supervisor",
    "first response time for a non delivery complaint",
    "refund credited in how many working days",
    "which customer exceeded the claim limit",
    "partner dispute window for cost allocation",
    "P2 priority trigger",
    "refund rate in Q1 2024",
]
docs = load_all(); chunks = chunk_docs(docs)      # the Lab 1 corpus, rebuilt
emb = get_embedder(); texts = [c.text for c in chunks]
if hasattr(emb, "fitted"):
    emb.fit(texts)                                # this embedder learns its vocabulary from the corpus
M = emb.encode(texts)      # M holds one row of numbers per piece of text
meta = [c.meta for c in chunks]                   # the labels that travel with each row of M
Q = emb.encode(QUERIES)    # the ten questions, turned into rows the same way
print("vectors:", M.shape, " queries:", Q.shape)
print("database versions:", vdb.versions())

## Step 1. The FreshCart pieces, in three real databases

**What you will do.** Run the next cell. It prints the size of the corpus, builds the answer key with
exact search (every question against every piece), then loads the same pieces into Chroma, Qdrant and
Milvus and asks each one the same ten questions.

**What you should see.** 30 vectors of 29 numbers each, about 3.4 KB. For "how many refund claims are
allowed in thirty days" exact search puts the 2022 policy first at 0.854, the 2024 policy second at
0.785, and the 2022 policy again third. All three databases return the same three pieces, and all
three score recall 1.000 over the ten questions.

**Why this step is here.** At 30 pieces every database is exact and instant, so the millisecond
columns prove nothing yet. Notice what the answer key itself puts first: the withdrawn 2022 policy.
The answer key tells you whether an index found what exact search would have found, not whether it
is the right document; that second question is Week 2. To see the databases differ, you need scale,
which is Step 2.

In [ ]:
# The answer key: exact search, every question against every piece, top ten kept.
print(f"{len(M):,} vectors of {M.shape[1]} numbers each, {M.nbytes/1024:.1f} KB raw")
truth = ground_truth(M, meta, Q, k=10)
flat = FlatStore(); flat.build(M, meta)
print("\nQ:", QUERIES[0])
for rank, (i, score) in enumerate(flat.search(Q[0], k=3), start=1):
    print(f"  exact search {rank}. score {score:.3f}  {meta[i].get('source','')}")

# The same pieces in the three databases. Each one keeps its files in a temporary folder.
small = []
for name in vdb.DATABASES:
    db = vdb.open_db(name); db.build(M); small.append(db)
print("\ntop 3 row numbers for that question:")
print(f"  {'exact search':13s}", [i for i, _ in flat.search(Q[0], k=3)])
for db in small:
    got = [db.search(q, 10) for q in Q]
    print(f"  {db.name:13s}", got[0][:3], f"  recall@10 over the ten questions: {vdb.recall([truth[i] for i in range(len(Q))], got):.3f}")
for db in small:
    db.remove()
print("\nAt this size every database is exact and instant. The trade only shows up at scale.")

## Step 2. Scale up to where speed differs

**What you will do.** Run the next cell. It makes 100,000 vectors of 128 numbers each, grouped in 1,000
clusters the way real embeddings are, plus 200 questions that sit near stored vectors. Then it builds
the answer key the only way that is always right: every question against every vector. Each vector
also gets a label from 0 to 9, which the filter test in Step 6 uses.

**What you should see.** 100,000 vectors, 51.2 MB of raw numbers, and the time exact search takes for
one question, a few milliseconds on a laptop.

**Why this step is here.** Exact search is the answer key for every number in this lab, and it is also
the slow method the databases exist to beat: its time grows with every row you add. Think of it as
the assistant reading all 100,000 past tickets for every customer message.

In [ ]:
# 100,000 vectors, 200 questions, and the exact top ten for each question.
bench = vdb.make_benchmark_set(n=100_000, dim=128, n_questions=200)
print(f"{len(bench.X):,} vectors of {bench.X.shape[1]} numbers, {bench.X.nbytes/1e6:.1f} MB raw")
print(f"{len(bench.Q)} questions; answer key built by exact search")
print(f"exact search, one question: {bench.exact_ms} ms. Every database below is measured against its answers.")

## Step 3. Load the three databases and measure them

**What you will do.** In the `YOUR CALL` cell name the database you expect to answer fastest, and the
recall you expect at the middle setting, ef 40. The next cell loads all 100,000 vectors into each
database, builds its index, and measures it two ways: one question at a time, and all 200 questions
sent in one call.

**What you should see.** Loading takes roughly 10 to 20 seconds per database on a laptop and about a
minute each on Databricks serverless. At ef 40 all three find 0.97 to 1.000 of the right answers. One
question at a time, Qdrant answers in under a millisecond, Chroma in under a millisecond on a laptop and
about 2 on Databricks, and Milvus in 25 to 80 milliseconds. Sent in one call, Milvus drops to between 0.5
and 2 milliseconds per question. On Databricks, two or three lines of gRPC messages ending in
`too_many_pings` may print while Milvus loads; they are harmless.

**Why this step is here.** Milvus's time is not its search. Milvus Lite, the small version
that runs inside a notebook, checks every stored row in Python before each search, so its time grows
with the number of rows; one call with 200 questions pays for that check once. The full Milvus server
does not work this way. That is a real lesson for choosing: the small local version of a database can
be much slower than the real one, so always time it the way your application will ask.

In [ ]:
# YOUR CALL
PREDICT_FASTEST = "qdrant"       # "chroma", "qdrant" or "milvus", one question at a time
PREDICT_RECALL = 0.90            # recall@10 you expect from the databases at ef 40

In [ ]:
# Load every database the same way (5,000 rows per call), then measure at ef 40.
dbs = []
for name in vdb.DATABASES:
    db = vdb.open_db(name)
    db.build(bench.X, bench.labels)
    dbs.append(db)
    print(f"{name:7s} loaded 100,000 vectors and built its index in {db.build_s} s")
rows = [vdb.measure(db, bench) for db in dbs]
print()
print(table(rows, ["database", "ef", "recall@10", "p50_ms", "p95_ms", "ms_each_in_one_call", "load_s"]))
fastest = min(rows, key=lambda r: r["p50_ms"])["database"]
print(f"\nyou said fastest: {PREDICT_FASTEST}, measured: {fastest}")
print(f"you said recall at ef 40: {PREDICT_RECALL}, measured: " + ", ".join(f"{r['database']} {r['recall@10']}" for r in rows))

## Step 4. Turn the dial and find where it stops paying

**What you will do.** `ef` is how many candidates the search keeps while it hops through the index.
In the `YOUR CALL` cell choose the settings to try and say where you think the curve flattens. The
next cell measures every setting on all three databases, and the one after it draws the three curves
and saves them to `out/lab2_curve.png`.

**What you should see.** Fifteen rows. Chroma climbs from about 0.80 at ef 10 to 0.99 or more by
ef 40 or 80. Qdrant and Milvus start higher, 0.92 to 0.94 at ef 10, and reach 0.99 or more by ef 40. The biggest
single gain arrives at ef 20 for all three. Time rises slowly with ef. Then a chart appears under the cell, with recall up the side and the middle response time along
the bottom, one line per database.

**Why this step is here.** The finding you take to an architect is this curve, not one number. It shows
that recall can be bought with search effort, and where buying more stops being worth it. Memory does
not move at all, because ef is used when a question is asked, not when the index is built. Chroma needs
one extra step to change ef: the collection is opened again, which `set_effort` does for you. Changing
it on an open collection silently does nothing.

In [ ]:
# YOUR CALL: the settings you want to sweep, and where you think the elbow is
SWEEP = [10, 20, 40, 80, 160]
PREDICT_ELBOW = 40

In [ ]:
# Sweep the one dial the three databases share. Recall uses all 200 questions; time uses
# the first 100, one at a time, so the slowest database does not hold the room up.
curve = []
for db in dbs:
    curve += vdb.sweep(db, bench, efforts=SWEEP)
print(table(curve, ["database", "ef", "recall@10", "p50_ms", "p95_ms"]))
print()
for db in dbs:
    pts = [r for r in curve if r["database"] == db.name]
    gains = [(pts[i]["recall@10"] - pts[i-1]["recall@10"], pts[i]["ef"]) for i in range(1, len(pts))]
    first_full = next((r["ef"] for r in pts if r["recall@10"] >= 0.995), None)
    print(f"{db.name:7s} biggest single gain arrives at ef {max(gains)[1] if gains else SWEEP[0]}; "
          f"it finds everything from ef {first_full}")
print(f"you predicted the elbow at ef {PREDICT_ELBOW}")
print("Memory does not move at all: ef is a query-time dial, not a build-time one.")

In [ ]:
# Draw the curves, not the numbers: each point is one ef setting, placed at the time it
# cost and the recall it bought. The shape is what you take to the memo.
import matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for db in dbs:
    pts = [r for r in curve if r["database"] == db.name]
    ax.plot([r["p50_ms"] for r in pts], [r["recall@10"] for r in pts], marker="o", label=db.name)
    for r in pts:
        if r is pts[0] or r is pts[-1]:     # label the two ends; the points between run in order
            ax.annotate(f"ef {r['ef']}", (r["p50_ms"], r["recall@10"]), textcoords="offset points", xytext=(4, -9), fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("p50 response time, milliseconds (log scale)"); ax.set_ylabel("recall@10"); ax.set_ylim(0.7, 1.02)
ax.set_title("Recall against time, one line per database, ef rising left to right")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
out = ROOT / "out"; out.mkdir(exist_ok=True)
fig.savefig(out / "lab2_curve.png", dpi=120)
print("saved", out / "lab2_curve.png")

## Step 5. Compression: what storing it smaller costs

**What you will do.** Compression stores each vector as short codes instead of full numbers, so the
index needs a fraction of the memory. Qdrant offers two kinds, and this cell builds its index three
ways: full numbers, int8 (each number squeezed into one byte, 4 times smaller) and product
quantization, or PQ (each vector cut into blocks, each block replaced by the number of its nearest
entry in a learned palette, 16 times smaller). Each compressed index is measured twice: on the codes
alone, and with the top candidates re-checked against the full numbers. Write your guess for PQ's
recall first.

**What you should see.** Five rows. Full numbers: recall 0.99 to 1.00. int8: about 0.91 on the codes
alone and 0.99 once re-checked. PQ: about 0.40 on the codes alone and about 0.58 re-checked. The
memory column falls from 51.2 MB to 12.8 MB and 3.2 MB.

**Why this step is here.** Compression is how very large collections fit in memory, and it is never
free. Re-checking the top candidates against the full numbers wins most of the recall back for int8.
PQ loses far more on these test vectors, which have only 128 numbers each; real embeddings of 1,536
numbers compress better, which is exactly why you measure on your own data before you switch it on.

In [ ]:
# YOUR CALL: recall@10 you expect from PQ, 16 times smaller, with no re-check
PREDICT_PQ_RECALL = 0.80

In [ ]:
# Qdrant three ways, each with and without re-checking. Takes about 30 seconds.
comp = vdb.compression_test(bench, ef=80)
print(table(comp, ["stored as", "vectors in memory, MB", "re-check", "recall@10", "p50_ms"]))
pq = next(r for r in comp if r["stored as"].startswith("PQ") and r["re-check"] == "no")
print(f"\nyou said PQ recall: {PREDICT_PQ_RECALL}, measured: {pq['recall@10']}")
print("The memory column is computed from the sizes: the folder on disk keeps the full numbers too, for the re-check.")

## Step 6. Test what a filter does to your results

**What you will do.** Ask for ten results among the vectors with label 3, which is one vector in ten,
and do it two ways. The database's own filter searches only inside the allowed rows. Filtering after
the search takes the ten nearest overall and then throws away what you may not see. Write your two
predictions in the `YOUR CALL` cell, then run the measurement.

**What you should see.** All three databases return 10 results with their own filter, and find the
right ones (recall 1.000 for Chroma and Qdrant, about 0.94 for Milvus). Filtering after the search
returns fewer than 1 result on average, and reports success anyway.

**Why this step is here.** This is the question that decides which product you can buy. A store that
can only filter after the search hands your users an almost empty answer and logs it as a success. In
Week 6 the same behaviour is what shows a document to the wrong person.

In [ ]:
# YOUR CALL: results returned when you ask for 10 inside a filter that keeps 1 row in 10
PREDICT_DB_FILTER = 10
PREDICT_AFTER = 5

In [ ]:
# The same 20 questions, label 3 only: the database's own filter, then a filter applied afterwards.
filt = [vdb.filter_test(db, bench, label=3) for db in dbs]
print(table(filt, ["database", "asked for", "database filter returned", "filter after the search returned", "recall inside the filter"]))
print(f"\nyou said: database filter {PREDICT_DB_FILTER}, after the search {PREDICT_AFTER}")
print("Filtering after the search returns an almost empty answer and reports success. Nothing errors.")

## Step 7. The dial on somebody else's service: Databricks AI Search

**What you will do.** Run the two cells. On Databricks, during class, the first one asks your trainer's
Databricks AI Search index the same 200 questions. That index holds the same 100,000 vectors, so your
answer key scores it exactly as it scored the other three. Anywhere else, or when the trainer's index
is not running, it says so and skips. The second cell prints which controls each option gives you,
and what you lose if somebody else picks ef for you.

**What you should see.** On Databricks: recall about 0.998 with no ef to turn, about 170 milliseconds
per question in the middle and under 400 at the slow end, and 10 of 10 results inside the filter. That
time is mostly the trip over the network, not the search itself. Then a table of dials, and your own sweep read at ef 10 and 20:
up to 2 of every 10 right answers lost at ef 10, with nothing on screen to tell you.

**Why this step is here.** You will be asked to use Databricks AI Search, Azure AI Search or pgvector,
and one of them may well be the right answer. The question is what you stop being able to control, and
what you therefore have to measure yourself. Do not create an AI Search endpoint of your own: each one
bills by the hour while it holds an index, and the trainer's shared one is all this step needs.

In [ ]:
# The trainer's shared index: same vectors, same questions, same answer key. Only on Databricks.
from managed_store import ai_search_measure, cost_of_their_default, DIALS, what_you_still_owe
managed = ai_search_measure(bench)
if managed:
    print(table([managed], list(managed.keys()), width=12))
    print("\nNo ef to turn: this is the recall the service picked for you, measured with your answer key.")

In [ ]:
# Which controls each option gives you. Read the last column twice: none of the managed
# services can tell you your recall, because recall needs your answer key, not theirs.
print(table(DIALS, list(DIALS[0].keys()), width=27))
print("\nIf somebody else had picked ef for you, from your own sweep:")
print(table(cost_of_their_default(curve, you_chose=80, their_defaults=(10, 20)),
            ["database", "setting", "recall@10", "answers you lose in 10"]))
print("\nWhat does not move to the vendor, whichever one you pick:")
for line in what_you_still_owe():
    print("  -", line)

### The question to answer

**Your corpus is 4 million chunks and growing, and the team wants a managed index by the end of the
quarter. Do you pick Databricks AI Search or one of the three you measured, and what do you build
anyway?**

There is a right shape of answer: pick the service, and keep the answer key and the release check
that the service cannot give you. Lab 8 is where that check becomes a gate.

## Step 8. Commit to an operating point, and pick one for the programme

**What you will do.** Write your service level in the `YOUR CALL` cell: the recall you need, the slow
end response time you will accept, and the database and setting you would ship. The next cell finds,
for each database, the cheapest ef that meets both targets, and lays the three side by side with the
things a benchmark does not measure.

**What you should see.** With the defaults of recall 0.95 and p95 under 5 milliseconds, Chroma and
Qdrant each meet the target somewhere between ef 20 and ef 80, and Milvus at no setting one question at a
time, because of the per-question check from Step 3. The table then adds load time, the filter result,
where each one runs, and its path to production.

**Why this step is here.** A measurement with no target is trivia. Writing the target first turns this
lab into a decision you can defend, and the syllabus asks for exactly that: benchmark three stores and
pick one for the programme. The store you pick is the one Days 3, 4, 9 and 11 lean on.

In [ ]:
# YOUR CALL: your service level, then the database and setting you would ship
TARGET_RECALL = 0.95
TARGET_P95_MS = 5.0
CHOSEN_DB = "qdrant"
CHOSEN_EF = 40

In [ ]:
# For each database, the cheapest setting on its curve that meets BOTH targets.
print(f"target: recall@10 >= {TARGET_RECALL} at p95 <= {TARGET_P95_MS} ms, one question at a time\n")
facts = {
    "chroma": ("yes", "yes, inside the notebook", "Chroma server or Chroma Cloud"),
    "qdrant": ("yes (beta edition)", "yes, inside the notebook", "Qdrant server or Qdrant Cloud"),
    "milvus": ("yes, install milvus-lite by name", "yes, inside the notebook", "Milvus server or Zilliz Cloud, same pymilvus code"),
}
pick = []
for db in dbs:
    ok = [r for r in curve if r["database"] == db.name and r["recall@10"] >= TARGET_RECALL and r["p95_ms"] <= TARGET_P95_MS]
    f = next(r for r in filt if r["database"] == db.name)
    win, dbx, prod = facts[db.name]
    pick.append({"database": db.name, "cheapest ef": ok[0]["ef"] if ok else "none",
                 "recall": ok[0]["recall@10"] if ok else "", "p95_ms": ok[0]["p95_ms"] if ok else "",
                 "load_s": db.build_s, "filter returned": f["database filter returned"],
                 "on Windows": win, "on Databricks": dbx, "path to production": prod})
if managed:
    pick.append({"database": "databricks ai search", "cheapest ef": "not yours", "recall": managed["recall@10"],
                 "p95_ms": managed["p95_ms"], "load_s": "trainer", "filter returned": managed["filter returned (of 10)"],
                 "on Windows": "no", "on Databricks": "yes, managed", "path to production": "it is the production service"})
print(table(pick, list(pick[0].keys()), width=14))
print(f"\nyou would ship: {CHOSEN_DB} at ef {CHOSEN_EF}")
print("Now say out loud what would change your mind: 10 million vectors, a strict filter, or an on-premises memory budget?")

## Step 9. Write the memo

**What you will do.** Run the cell, copy the measured lines into the **Your write-up** cell at the bottom
of this notebook, and add the two paragraphs it asks for.

**What you should see.** One line per database with its recall, both response times and load time,
then the filter result, the compression result and your chosen operating point.

**Why this step is here.** The memo is the deliverable of the day. The numbers are the easy half. The
sentence that matters is the one saying what would change your mind, because that is what makes this a
decision rather than an opinion.

In [ ]:
# Collect every number you measured, formatted so it can go straight into the memo.
print("paste these measured lines into the Your write-up cell at the bottom\n")
print(f"- data searched: {len(bench.X):,} vectors of {bench.X.shape[1]} numbers, {bench.X.nbytes/1e6:.1f} MB raw; "
      f"exact search {bench.exact_ms} ms per question")
for r in rows:
    print(f"- {r['database']} at ef {r['ef']}: recall@10 {r['recall@10']}, p50 {r['p50_ms']} ms, p95 {r['p95_ms']} ms, "
          f"{r['ms_each_in_one_call']} ms each in one call, load {r['load_s']} s")
if managed:
    print(f"- databricks ai search: recall@10 {managed['recall@10']}, p50 {managed['p50_ms']} ms, p95 {managed['p95_ms']} ms")
for f in filt:
    print(f"- filter test, {f['database']}: its own filter returned {f['database filter returned']} of 10, "
          f"filtering after the search returned {f['filter after the search returned']}")
print("- compression (Qdrant, ef 80): " + "; ".join(f"{r['stored as']} re-check {r['re-check']}: {r['recall@10']}" for r in comp))
print(f"- operating point: {CHOSEN_DB} at ef {CHOSEN_EF}, for recall >= {TARGET_RECALL} at p95 <= {TARGET_P95_MS} ms")
print("\nThen write the two paragraphs nobody can generate for you: why this database here, and what would change your mind.")
for db in dbs:
    db.remove()      # delete the three databases' temporary folders

## When you are done

- The Azure version measures the same questions against Azure AI Search, with exhaustive search
  as the ground truth: `python labs\lab02_azure.py --azure`. Add `--pgvector` to measure PostgreSQL
  with pgvector too, if your trainer has set it up. These are the two stores the syllabus names.
- Keep `out/lab2_curve.png`. It goes in the memo.

## Your write-up

**Type your answers into this cell.** Double click it to edit, then click away to render it again.
Everything you write here leaves with the notebook when you export it, and that export is what you
hand in.

Do not write in `handouts/lab2_memo.md`. That folder is rebuilt from the lab pack every session, so anything you
type there is gone tomorrow.

---

**Pair:**

### Which database, and at what operating point

### The curve behind that choice

| ef | Chroma recall@10 | Qdrant recall@10 | Milvus recall@10 | p50 ms of your choice |
|---|---|---|---|---|
| 10 | | | | |
| 20 | | | | |
| 40 | | | | |
| 80 | | | | |
| 160 | | | | |

Where is the elbow?

### How each database handled the selective filter

Its own filter returned ___ results. Filtering after the search returned ___ results (asked for 10).

Why does that matter in production?

### What compression would cost you

### The operational risk you are accepting

### What would change your mind